# Exercise 4 - Dynamic Programming
An organisation prepares a shipment of supplies of four different types: A, B, C, D

Each type has three different alternative packages, which differ in **weight** and **profit**
The maximum weight is 13.

Select packages to maximise the profit

## Question 15
Stages:

States:

## Question 16
Forward recursion for dynamic program:
- $d_A(S) = max\{\, b_i^A: \quad \forall i=1,2,3: w_i^A \leq S, -\infty\}$
- $d_B(S) = max\{\, d_A(S-w_i^B) + b_i^B: \quad \forall i=1,2,3: w_i^B \leq S, -\infty\}$
- $d_C(S) = max\{\, d_B(S-w_i^C) + b_i^C: \quad \forall i=1,2,3: w_i^C \leq S, -\infty\}$
- $d_D(S) = max\{\, d_C(S-w_i^D) + b_i^D: \quad \forall i=1,2,3: w_i^D \leq S, -\infty\}$

-> with $b_i^L$ is the **profit** of the $i$-th package of type $L \in \{A,B,C,D\}$

-> with $w_i^L$ is the **weight** of the $i$-th package of type $L \in \{A,B,C,D\}$

## Question 17
solve the dynamic program


In [14]:
# Question 17: dynamic program
using JuMP, GLPK

global w = [           # weight
    [2,3,5],
    [2,4,5],
    [1,3,4],
    [2,3,5]
]
global b = [           # benefit
    [10,14,20], #A = 1
    [8,15,18],  #B = 2
    [5,12,16],  #C = 3
    [7,11,19]   #D = 4
]

global capacity = 13     # total capacity

D = fill(-Inf, 4, capacity + 1)     # saves 'already-calculated' results
X = Array{Vector{Vector{Int}}}(undef, 4, capacity + 1)  # saves which package was chosen each

function d(n,S)
    if n == 1
        value = -Inf
        best_x = Vector{Vector{Int}}()

        for i = 1:3
            if w[n][i] <= S && b[n][i] > value
                value = b[n][i]
                x_now = [0,0,0]
                x_now[i] = 1
                best_x = [x_now]
            end
        end

    else
        value = -Inf
        best_x = Vector{Vector{Int}}()

        for i = 1:3
            if w[n][i] <= S

                prev_value = D[n-1, S-w[n][i]+1]

                if prev_value > -Inf
                    candidate = prev_value + b[n][i]

                    if candidate > value
                        value = candidate

                        x_now = [0,0,0]
                        x_now[i] = 1

                        best_x = vcat(
                            deepcopy(X[n-1, S-w[n][i]+1]),
                            [x_now]
                        )
                    end
                end
            end
        end
    end

    return value, best_x
end

# compute maximum profit for every storage-value and save the max-Profit as well as the solution for the profit
for n=1:4
    for S=0:capacity
        value, x = d(n,S)
        D[n,S+1] = value
        X[n,S+1] = x
    end
end

# PRINT all solutions:
for n in 1:4
    println("Stage n = ", n)

    for S in 0:capacity
        print(
            "S = ", S,
            "   value = ", D[n,S+1])
        if isfinite(D[n,S+1])
            print("   solution = ", X[n,S+1])
        end
        println()
    end

    println()
end

# print maximum profit
println("maximum profit:     ", D[4,13+1])
println("solution to profit: ",X[4,13+1])

Stage n = 1
S = 0   value = -Inf
S = 1   value = -Inf
S = 2   value = 10.0   solution = [[1, 0, 0]]
S = 3   value = 14.0   solution = [[0, 1, 0]]
S = 4   value = 14.0   solution = [[0, 1, 0]]
S = 5   value = 20.0   solution = [[0, 0, 1]]
S = 6   value = 20.0   solution = [[0, 0, 1]]
S = 7   value = 20.0   solution = [[0, 0, 1]]
S = 8   value = 20.0   solution = [[0, 0, 1]]
S = 9   value = 20.0   solution = [[0, 0, 1]]
S = 10   value = 20.0   solution = [[0, 0, 1]]
S = 11   value = 20.0   solution = [[0, 0, 1]]
S = 12   value = 20.0   solution = [[0, 0, 1]]
S = 13   value = 20.0   solution = [[0, 0, 1]]

Stage n = 2
S = 0   value = -Inf
S = 1   value = -Inf
S = 2   value = -Inf
S = 3   value = -Inf
S = 4   value = 18.0   solution = [[1, 0, 0], [1, 0, 0]]
S = 5   value = 22.0   solution = [[0, 1, 0], [1, 0, 0]]
S = 6   value = 25.0   solution = [[1, 0, 0], [0, 1, 0]]
S = 7   value = 29.0   solution = [[0, 1, 0], [0, 1, 0]]
S = 8   value = 32.0   solution = [[0, 1, 0], [0, 0, 1]]
S = 9   

## Question 18
Formulate the optimization problem as a binary integer program and solve it using Julia/JuMP. 


Compare the solution obtained with the one obtained via dynamic programming

*$\rightarrow$ we got **53** as optimal solution both times, but different packages were chosen (no unique optimal solution)*

In [4]:
# Question 18: binary integer program
using JuMP, GLPK

model = Model(GLPK.Optimizer)

p = [10 14 20;
      8 15 18;
      5 12 16;
      7 11 19]

a = [2 3 5;
     2 4 5;
     1 3 4;
     2 3 5]

@variable(model, x[1:4, 1:3], Bin)

# Maximize total medical benefit
@objective(model, Max, sum(p[i,j] * x[i,j] for i in 1:4, j in 1:3))

# Exactly one package from each category
@constraint(model, [i in 1:4], sum(x[i,j] for j in 1:3) == 1)

# Aircraft capacity
@constraint(model, sum(a[i,j] * x[i,j] for i in 1:4, j in 1:3) <= 13)

# Solve
optimize!(model)

if termination_status(model) == MOI.OPTIMAL
    println("Objective value = ", objective_value(model))

    for i in 1:4
        for j in 1:3
            if value(x[i,j]) > 0.5
                println("Selected package: ", Char(Int('A') + i - 1), j)
            end
        end
    end
else
    println("Optimization was not successful: ", termination_status(model))
end

Objective value = 53.0
Selected package: A1
Selected package: B1
Selected package: C3
Selected package: D3
